In [1]:
import sys
import re
from pathlib import Path

import pandas as pd

BASE_DIR = Path().resolve()
CSV_PATH = BASE_DIR / "atletas_modalidades_ndu.csv"

# Ajusta o import para conseguir importar o módulo do mesmo diretório
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

import extracao_dados


def load_atletas_df():
    """Carrega o CSV de atletas/modalidades ou executa a extração se não existir."""
    if not CSV_PATH.exists():
        print("Arquivo CSV não encontrado. Extraindo dados do site...")
        extracao_dados.main()

    df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
    return df


def atletas_por_modalidades(modalidades, df=None):
    """Retorna todos os atletas que estão inscritos em pelo menos uma das modalidades fornecidas."""
    if df is None:
        df = load_atletas_df()

    modalidades = [m.strip().lower() for m in modalidades if m and m.strip()]
    if not modalidades:
        raise ValueError("Informe ao menos uma modalidade.")

    def tem_modalidade(texto):
        if pd.isna(texto):
            return False
        texto = str(texto).lower()
        return any(mod in texto for mod in modalidades)

    resultado = df[df["modalidades_cadastradas"].apply(tem_modalidade)].copy()
    resultado = resultado.sort_values(by=["nome", "modalidades_cadastradas"]).reset_index(drop=True)
    return resultado


def imprime_lista_por_modalidades(modalidades, df=None):
    """Imprime a modalidade e os nomes dos atletas inscritos, numerados."""
    if df is None:
        df = load_atletas_df()

    modalidades = [m.strip() for m in modalidades if m and m.strip()]
    if not modalidades:
        raise ValueError("Informe ao menos uma modalidade.")

    for modalidade in modalidades:
        termo = modalidade.lower()
        df_mod = df[df["modalidades_cadastradas"].str.lower().str.contains(re.escape(termo), na=False)].copy()
        df_mod = df_mod.sort_values(by="nome").reset_index(drop=True)

        print(modalidade)
        if df_mod.empty:
            print("  Nenhum atleta encontrado.")
        else:
            for idx, nome in enumerate(df_mod["nome"], start=1):
                print(f"  {idx}. {nome}")
        print()


# Exemplo de uso:
modalidades_da_semana = ["Basquete Feminino", "Voleibol Masculino", "Futsal Feminino", "Basquete Masculino"]
imprime_lista_por_modalidades(modalidades_da_semana)


Basquete Feminino
  1. Anny Alves Marcelino
  2. Beatriz Bobrow Bozzo Laginestra Carlos
  3. Beatriz Hyde
  4. Camila de Assis Moura Tavares
  5. Catarine Pinter Marcos
  6. Daniela Sanchez
  7. Gabriela de Mesquita Sampaio Duarte
  8. Giulia Englerth de Sá Gramigna
  9. Ingrid Karoliny Machado de Sousa
  10. Julia Benatti Ribeiro
  11. Lani Kristhel De Oliveira Baptista
  12. Layne Pereira da Silva
  13. Luana Prado Lopes Guimarães
  14. Luma Bernardelli Fadel
  15. Maria Giovanna Fenuchi Loyola
  16. Maria Sofia Daud Malouf
  17. Marina Abib Fernandes de Barros
  18. Pietra Macedo de Souza
  19. Rafaela Coelho Tosetto Fagundes
  20. Sophia Tuma
  21. Vitória Veludo dos Santos

Voleibol Masculino
  1. Alberto Martins Pereira Carrera
  2. Arthur Silva Fedato
  3. Felipe Castelar Do Prado Silva
  4. Felipe De Paula Eduardo
  5. Fernando Fiore
  6. Francisco Kalmar Hubner Marques
  7. Gabriel Fuks Ribeiro
  8. Gabriel Rodrigues Nascimento Cunha
  9. Gustavo Doniani Lagôa Gomes
  10. Gust